# AML Benchmark — análise comparativa de desempenho

Mede e compara as várias fases do pipeline:

- **Qualidade do modelo** (accuracy, precision, recall, F1) sobre as previsões com `truth` conhecida.
- **Throughput** do consumer streaming (msgs/s) a partir das janelas de métricas escritas pelo consumer.
- **Tempo de leitura batch** dos datasets `LI-Small_Trans.csv` vs `LI-Medium_Trans.csv` —
  comparação direta do efeito do volume.
- **Distribuição de classificações** (sanity check / matriz de confusão).

In [ ]:
import time
from pyspark.sql import SparkSession, functions as F

HDFS_BASE   = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project"
PRED_PATH   = f"{HDFS_BASE}/stream/predictions_kafka"
METRICS_PATH= f"{HDFS_BASE}/stream/metrics_kafka"
DS_SMALL    = f"{HDFS_BASE}/dataset/LI-Small_Trans.csv"
DS_MEDIUM   = f"{HDFS_BASE}/dataset/LI-Medium_Trans.csv"

spark = SparkSession.builder.appName("AML Benchmark").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

## 1) Qualidade do modelo

Como as mensagens trazem `Is Laundering` da fonte original, podemos calcular métricas reais
comparando previsão vs verdade. Numa demo "real" o `truth` não viria — usamos só para validação.

In [ ]:
df = spark.read.parquet(PRED_PATH)
n = df.count()
if n == 0:
    print("0 previsões disponíveis. Corre o producer + consumer primeiro.")
else:
    m = df.agg(
        F.sum(F.when((F.col("prediction") == 1) & (F.col("truth") == 1), 1).otherwise(0)).alias("tp"),
        F.sum(F.when((F.col("prediction") == 1) & (F.col("truth") == 0), 1).otherwise(0)).alias("fp"),
        F.sum(F.when((F.col("prediction") == 0) & (F.col("truth") == 1), 1).otherwise(0)).alias("fn"),
        F.sum(F.when((F.col("prediction") == 0) & (F.col("truth") == 0), 1).otherwise(0)).alias("tn"),
    ).collect()[0]
    tp, fp, fn, tn = m["tp"] or 0, m["fp"] or 0, m["fn"] or 0, m["tn"] or 0
    acc  = (tp + tn) / n
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec  = tp / (tp + fn) if (tp + fn) else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    print(f"n={n:,}  TP={tp:,} FP={fp:,} FN={fn:,} TN={tn:,}")
    print(f"accuracy={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")

## 2) Throughput do consumer streaming

Lê as métricas escritas em janelas de 30s pelo `aml_kafka_consumer` e calcula throughput médio/pico.

In [ ]:
try:
    m = spark.read.parquet(METRICS_PATH)
    if m.count() == 0:
        print("sem janelas registadas.")
    else:
        summary = m.agg(
            F.avg("n_msgs").alias("avg_per_window"),
            F.max("n_msgs").alias("peak_per_window"),
            F.sum("n_msgs").alias("total"),
            F.count("*").alias("windows"),
        ).collect()[0]
        # janelas são de 30s
        avg_per_sec  = (summary["avg_per_window"]  or 0) / 30.0
        peak_per_sec = (summary["peak_per_window"] or 0) / 30.0
        print(f"janelas={summary['windows']}  total_msgs={summary['total']:,}")
        print(f"throughput médio = {avg_per_sec:.0f} msgs/s   pico = {peak_per_sec:.0f} msgs/s")
except Exception as e:
    print(f"sem métricas em {METRICS_PATH}: {type(e).__name__}")

## 3) Tempo batch — Small vs Medium

Mede o tempo de leitura+contagem de cada dataset. Comparação direta do efeito do volume.

In [ ]:
def batch_read_time(path, label):
    t0 = time.time()
    n = spark.read.option("header", True).option("inferSchema", False).csv(path).count()
    dt = time.time() - t0
    print(f"[{label}] {n:>12,} linhas em {dt:6.1f}s  ({n/dt:>10,.0f} linhas/s)")
    return n, dt

n_s, t_s = batch_read_time(DS_SMALL,  "Small ")
n_m, t_m = batch_read_time(DS_MEDIUM, "Medium")
print(f"\nMedium / Small: {n_m/n_s:.1f}x linhas, {t_m/t_s:.1f}x tempo")

## 4) Distribuição de classificações (matriz de confusão)

In [ ]:
try:
    df = spark.read.parquet(PRED_PATH)
    df.groupBy("prediction", "truth").count().orderBy("prediction", "truth").show()
except Exception as e:
    print(f"sem previsões: {type(e).__name__}")